# Habita MX · CrewAI + Ollama
Equipo 6 · Universidad Anáhuac Puebla

Ejecuta las celdas en orden dentro del ambiente del proyecto. Son cuatro agentes reales y datos simulados. Built with Llama. Los archivos oficiales del proyecto están en los módulos de `habita/`.

In [ ]:
import sys, json, asyncio
from pathlib import Path
from IPython.display import display, Markdown
from habita.aplicacion import RAIZ, diagnosticar, ejecutar, reporte_markdown
print('Python:', sys.executable)
print('Proyecto:', RAIZ)

## 1. Comprobar el modelo local
Ollama debe estar abierto. El diagnóstico verifica el nombre del modelo y registra su digest.

In [ ]:
diagnostico = diagnosticar()
diagnostico

## 2. Revisar y editar el perfil
Edita valores del diccionario antes de ejecutar. Los seis pesos deben sumar 1. Todos los montos son MXN. La descripción libre no reemplaza las cifras estructuradas.

In [ ]:
perfil = json.loads((RAIZ / 'datos/perfil_demo.json').read_text(encoding='utf-8'))
perfil

In [ ]:
# Ejemplo opcional: perfil['pago_maximo_usuario'] = 12000
# Se guarda una copia y se conserva el caso original.
perfil_notebook = RAIZ / 'datos/perfil_notebook.json'
perfil_notebook.write_text(json.dumps(perfil, ensure_ascii=False, indent=2), encoding='utf-8')
print(perfil_notebook)

## 3. Ejecutar los cuatro agentes
Puede tardar desde decenas de segundos hasta varios minutos. Las líneas impresas muestran herramientas usadas y tareas aceptadas. Si falla, revisa `error.json`; no se sustituye por una simulación.

In [ ]:
# Ejecutar en otro hilo mantiene compatible el ciclo de eventos de Jupyter.
resultado, carpeta = await asyncio.to_thread(ejecutar, perfil_path=perfil_notebook, modo='crew')
display(Markdown(reporte_markdown(resultado)))
print('Evidencia:', carpeta)

## 4. Revisar la colaboración
El log contiene acciones observables y resúmenes finales, no cadenas internas de pensamiento.

In [ ]:
eventos = [json.loads(l) for l in (carpeta / 'interacciones.jsonl').read_text(encoding='utf-8').splitlines()]
for evento in eventos:
    if evento['evento'] == 'tarea_completada':
        print(evento['etapa'], '<-', evento['contexto'])
        print(evento['conclusion']['resumen'], '\n')

## 5. Contrastar con el motor de cálculo
Esta línea base no ejecuta IA. Debe coincidir con los datos oficiales del Crew.

In [ ]:
base, carpeta_base = ejecutar(perfil_path=perfil_notebook, modo='calculo')
assert resultado['resultado'] == base['resultado']
print('Ranking y cifras coinciden con la línea base.')

## 6. Preguntas para la exposición
- ¿Por qué Z02 puede quedar arriba aunque Z01 tenga menor mensualidad?
- ¿Qué ocurre si el pago máximo baja a $10,000?
- ¿Por qué Z06 no puede recomendarse?
- ¿Qué hace el guardrail si el modelo inventa un ID?
- ¿Qué métricas de la Fase 2 aún no han sido evaluadas?

Consulta `docs/Fase_3.md` y `README.md` para especificación y referencias.